In [6]:
import sys
import os

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project Root:", PROJECT_ROOT)

Project Root: c:\Users\sanka\OneDrive\Desktop\mulitmodal stress detection


In [ ]:
import time
import pickle
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score
)
from torch.utils.data import DataLoader
from models.cnn_encoder import MultiModalCNN
from experiments.train_cnn_loso import (
    SUBJECTS,
    WESADDataset,
    load_subject,
    load_all_except,
    get_val_subject,
    train_one_fold
)

device = torch.device("cpu")

torch.set_num_threads(12)
torch.manual_seed(42)
np.random.seed(42)

os.makedirs("../results/models", exist_ok=True)

all_results = []
all_predictions = {}

start_total = time.time()

print(f"Subjects: {SUBJECTS}")
print(f"Number of folds: {len(SUBJECTS)}")

for fold_idx, test_sid in enumerate(SUBJECTS):

    print("\n" + "=" * 60)
    print(f"Fold {fold_idx + 1}/{len(SUBJECTS)}")
    print(f"Test Subject: S{test_sid}")
    print("=" * 60)

    checkpoint_path = f"../results/models/cnn_S{test_sid}.pt"

    if os.path.exists(checkpoint_path):
        print(f"Skipping S{test_sid} (already completed)")
        continue

    val_sid = get_val_subject(test_sid)

    torch.manual_seed(42)
    np.random.seed(42)

    train_X, train_y = load_all_except(test_sid, val_sid)
    val_X, val_y = load_subject(val_sid)
    test_X, test_y = load_subject(test_sid)

    print(
        f"Train={len(train_y)} | "
        f"Val={len(val_y)} | "
        f"Test={len(test_y)}"
    )

    print("Train shape:", train_X.shape)
    print("Val shape:", val_X.shape)
    print("Test shape:", test_X.shape)

    train_loader = DataLoader(
        WESADDataset(train_X, train_y),
        batch_size=32,
        shuffle=True
    )
    val_loader = DataLoader(
        WESADDataset(val_X, val_y),
        batch_size=32,
        shuffle=False
    )
    test_loader = DataLoader(
        WESADDataset(test_X, test_y),
        batch_size=32,
        shuffle=False
    )
    model = MultiModalCNN(
        embed_dim=128,
        n_modalities=4,
        n_classes=2
    ).to(device)

    print("Using embed_dim=128")
    fold_start = time.time()

    model, best_val_f1 = train_one_fold(
        model,
        train_loader,
        val_loader,
        device,
        epochs=50,
        patience=10
    )

    fold_time = time.time() - fold_start
    model.eval()

    test_preds = []
    test_probs = []
    test_true = []

    with torch.no_grad():

        for Xb, yb in test_loader:

            Xb = Xb.to(device)

            logits = model(Xb)

            probs = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            preds = logits.argmax(dim=1)

            test_preds.extend(
                preds.cpu().numpy()
            )

            test_probs.extend(
                probs.cpu().numpy()
            )

            test_true.extend(
                yb.numpy()
            )

    acc = accuracy_score(
        test_true,
        test_preds
    )

    f1 = f1_score(
        test_true,
        test_preds,
        average="macro",
        zero_division=0
    )

    try:
        auroc = roc_auc_score(
            test_true,
            test_probs
        )
    except ValueError:
        auroc = np.nan

    all_results.append({
        "subject": test_sid,
        "val_subject": val_sid,
        "val_f1": best_val_f1,
        "accuracy": acc,
        "f1_macro": f1,
        "auroc": auroc,
        "train_time_sec": fold_time
    })

    all_predictions[test_sid] = {
        "y_true": np.array(test_true),
        "y_pred": np.array(test_preds),
        "y_prob": np.array(test_probs)
    }
    torch.save(
        model.state_dict(),
        checkpoint_path
    )
    pd.DataFrame(all_results).to_csv(
        "../results/cnn_full_modality_results.csv",
        index=False
    )

    with open(
        "../results/cnn_predictions_full.pkl",
        "wb"
    ) as f:
        pickle.dump(
            all_predictions,
            f
        )

    print(
        f"S{test_sid} | "
        f"ValF1={best_val_f1:.4f} | "
        f"TestF1={f1:.4f} | "
        f"AUROC={auroc:.4f} | "
        f"Time={fold_time/60:.1f} min"
    )

results_path = "../results/cnn_full_modality_results.csv"

if os.path.exists(results_path):

    results_df = pd.read_csv(results_path)
    mean_acc = results_df["accuracy"].mean()
    std_acc = results_df["accuracy"].std()
    mean_f1 = results_df["f1_macro"].mean()
    std_f1 = results_df["f1_macro"].std()
    mean_auroc = results_df["auroc"].mean()
    std_auroc = results_df["auroc"].std()
    total_time = time.time() - start_total

    print("\n" + "=" * 60)
    print("FINAL RESULTS")
    print("=" * 60)
    print(f"Accuracy : {mean_acc:.4f} ± {std_acc:.4f}")
    print(f"Macro F1 : {mean_f1:.4f} ± {std_f1:.4f}")
    print(f"AUROC    : {mean_auroc:.4f} ± {std_auroc:.4f}")
    print(f"Total Time: {total_time/60:.1f} minutes")

Subjects: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]
Number of folds: 15

Fold 1/15
Test Subject: S2
Train=1897 | Val=150 | Test=140
Train shape: (1897, 4, 3000)
Val shape: (150, 4, 3000)
Test shape: (140, 4, 3000)
Using embed_dim=128
S2 | ValF1=0.9923 | TestF1=0.7604 | AUROC=0.8955 | Time=20.7 min

Fold 2/15
Test Subject: S3
Train=1895 | Val=150 | Test=142
Train shape: (1895, 4, 3000)
Val shape: (150, 4, 3000)
Test shape: (142, 4, 3000)
Using embed_dim=128
S3 | ValF1=1.0000 | TestF1=0.7654 | AUROC=0.9621 | Time=3.0 min

Fold 3/15
Test Subject: S4
Train=1894 | Val=150 | Test=143
Train shape: (1894, 4, 3000)
Val shape: (150, 4, 3000)
Test shape: (143, 4, 3000)
Using embed_dim=128
S4 | ValF1=1.0000 | TestF1=0.6961 | AUROC=0.9993 | Time=5.3 min

Fold 4/15
Test Subject: S5
Train=1891 | Val=150 | Test=146
Train shape: (1891, 4, 3000)
Val shape: (150, 4, 3000)
Test shape: (146, 4, 3000)
Using embed_dim=128
S5 | ValF1=0.9923 | TestF1=0.9111 | AUROC=0.9906 | Time=11.7 min

Fold 5/15


In [8]:
results_df = pd.DataFrame(all_results)
print(results_df)

print("\n--- CNN Full-Modality Summary ---")
print(f"Mean Accuracy: {results_df['accuracy'].mean():.4f} ± {results_df['accuracy'].std():.4f}")
print(f"Mean Macro-F1: {results_df['f1_macro'].mean():.4f} ± {results_df['f1_macro'].std():.4f}")
print(f"Mean AUROC:    {results_df['auroc'].mean():.4f} ± {results_df['auroc'].std():.4f}")

results_df.to_csv("../results/cnn_full_modality_results.csv", index=False)

    subject  val_subject    val_f1  accuracy  f1_macro     auroc  \
0         2           17  0.992298  0.771429  0.760428  0.895541   
1         3           17  1.000000  0.838028  0.765424  0.962143   
2         4           17  1.000000  0.811189  0.696104  0.999283   
3         5           17  0.992298  0.931507  0.911062  0.990614   
4         6           17  0.992298  0.889655  0.861905  0.967168   
5         7           17  1.000000  0.993103  0.991560  1.000000   
6         8           17  0.984681  0.904110  0.894530  0.999109   
7         9           17  1.000000  0.710345  0.437361  0.849977   
8        10           17  0.992298  0.906667  0.882524  0.987812   
9        11           17  0.984681  0.979592  0.976408  0.999782   
10       13           17  0.992208  0.938776  0.930019  0.997540   
11       14           17  0.984681  0.972789  0.968723  0.994771   
12       15           17  0.984507  0.816327  0.739036  0.921227   
13       16           17  1.000000  0.972789  0.

In [9]:
import pickle

with open("../results/cnn_predictions_full.pkl", "wb") as f:
    pickle.dump(all_predictions, f)

print("Saved predictions for all 15 folds.")
print("Per-fold model checkpoints saved individually in ../results/models/cnn_S{2..11,13..17}.pt")

Saved predictions for all 15 folds.
Per-fold model checkpoints saved individually in ../results/models/cnn_S{2..11,13..17}.pt


In [ ]:
import pandas as pd

cnn_results = pd.read_csv("../results/cnn_full_modality_results.csv")
all_handcrafted = pd.read_csv("../results/handcrafted_loso_full.csv")

svm_results = all_handcrafted[
    all_handcrafted["classifier"] == "SVM"
].copy()

comparison_df = pd.merge(
    svm_results[["subject", "accuracy", "macro_f1", "auroc"]],
    cnn_results[["subject", "accuracy", "f1_macro", "auroc"]],
    on="subject",
    suffixes=("_svm", "_cnn")
)
comparison_df["f1_delta"] = (
    comparison_df["f1_macro"] -
    comparison_df["macro_f1"]
)
comparison_df["accuracy_delta"] = (
    comparison_df["accuracy_cnn"] -
    comparison_df["accuracy_svm"]
)
comparison_df["auroc_delta"] = (
    comparison_df["auroc_cnn"] -
    comparison_df["auroc_svm"]
)

comparison_df = comparison_df.sort_values(
    "f1_delta",
    ascending=False
)
print("=" * 80)
print("CNN vs SVM (Subject-wise)")
print("=" * 80)
print(
    comparison_df[
        [
            "subject",
            "macro_f1",
            "f1_macro",
            "f1_delta",
            "accuracy_svm",
            "accuracy_cnn",
            "auroc_svm",
            "auroc_cnn"
        ]
    ].round(4)
)

svm_f1 = svm_results["macro_f1"].mean()
cnn_f1 = cnn_results["f1_macro"].mean()
svm_acc = svm_results["accuracy"].mean()
cnn_acc = cnn_results["accuracy"].mean()
svm_auroc = svm_results["auroc"].mean()
cnn_auroc = cnn_results["auroc"].mean()

print("\n" + "=" * 80)
print("OVERALL COMPARISON")
print("=" * 80)
print(f"SVM Mean F1       : {svm_f1:.4f}")
print(f"CNN Mean F1       : {cnn_f1:.4f}")
print(f"F1 Delta          : {cnn_f1 - svm_f1:.4f}")
print()
print(f"SVM Mean Accuracy : {svm_acc:.4f}")
print(f"CNN Mean Accuracy : {cnn_acc:.4f}")
print()
print(f"SVM Mean AUROC    : {svm_auroc:.4f}")
print(f"CNN Mean AUROC    : {cnn_auroc:.4f}")

cnn_wins = (comparison_df["f1_delta"] > 0).sum()
svm_wins = (comparison_df["f1_delta"] < 0).sum()
ties = (comparison_df["f1_delta"] == 0).sum()

print("\n" + "=" * 80)
print("WIN COUNT")
print("=" * 80)

print(f"CNN wins on {cnn_wins} subjects")
print(f"SVM wins on {svm_wins} subjects")
print(f"Ties: {ties}")

best_subject = comparison_df.iloc[0]
worst_subject = comparison_df.iloc[-1]

print("\n" + "=" * 80)
print("MOST IMPROVED SUBJECT")
print("=" * 80)

print(
    f"S{int(best_subject['subject'])}: "
    f"Delta = {best_subject['f1_delta']:.4f}"
)

print("\n" + "=" * 80)
print("WORST SUBJECT")
print("=" * 80)

print(
    f"S{int(worst_subject['subject'])}: "
    f"Delta = {worst_subject['f1_delta']:.4f}"
)

CNN vs SVM (Subject-wise)
    subject  macro_f1  f1_macro  f1_delta  accuracy_svm  accuracy_cnn  \
3         5    0.6995    0.9111    0.2115        0.8014        0.9315   
10       13    0.7350    0.9300    0.1950        0.7415        0.9388   
9        11    0.8480    0.9764    0.1284        0.8571        0.9796   
5         7    0.8707    0.9916    0.1209        0.9034        0.9931   
11       14    0.8615    0.9687    0.1072        0.8707        0.9728   
13       16    0.9099    0.9687    0.0588        0.9184        0.9728   
14       17    0.9478    0.9535    0.0058        0.9533        0.9600   
4         6    0.8903    0.8619   -0.0284        0.9103        0.8897   
6         8    0.9358    0.8945   -0.0412        0.9452        0.9041   
8        10    0.9545    0.8825   -0.0720        0.9600        0.9067   
0         2    0.8425    0.7604   -0.0821        0.8714        0.7714   
1         3    0.8617    0.7654   -0.0963        0.8944        0.8380   
12       15    0.8778    